
# Dataset 1 — Structured Data Parsing

## Project
AI-ML Integrated SOC Platform

## Phase
02 — Structured Data Parsing

## Objective
To inspect, parse, validate and standardize nested malware intelligence fields in Dataset 1.

## Input Dataset
Consolidated Dataset 1

- Records: 10,000
- Columns: 32
- Unique SHA256: 10,000

## Main Tasks
1. Load the consolidated dataset.
2. Inspect nested and serialized fields.
3. Identify JSON and Python literal representations.
4. Parse structured values safely.
5. Handle malformed and unparseable values.
6. Validate parsed structures.
7. Preserve original evidence.
8. Save the parsed dataset and parsing audit.

## Important Rule
Never overwrite the original raw or consolidated datasets.

## Expected Output
A validated structured dataset ready for the next data-cleaning phase.


Step 2. Load the consolidated dataset

In [18]:

import pandas as pd
import numpy as np
import json
import ast
import os
from pathlib import Path
from collections import Counter

# Locate the consolidated dataset
filename = "dataset_1_consolidated_10000.pkl"

possible_paths = [
    Path("data/processed") / filename,
    Path("../data/processed") / filename
]

dataset_path = next(
    (p for p in possible_paths if p.exists()),
    None
)

if dataset_path is None:
    raise FileNotFoundError(
        f"Could not find {filename}. Check data/processed folder."
    )

# Load dataset
df = pd.read_pickle(dataset_path)

# Create independent working DataFrame
df = df.copy()

print("DATASET LOADED SUCCESSFULLY")
print("=" * 50)

print("Dataset shape:", df.shape)
print("Total records:", len(df))
print("Unique SHA256:", df["sha256"].nunique())
print("Duplicate SHA256:", df["sha256"].duplicated().sum())

print("\nLabel distribution:")
print(df["label"].value_counts())

print("\nColumns:")
print(df.columns.tolist())


DATASET LOADED SUCCESSFULLY
Dataset shape: (10000, 32)
Total records: 10000
Unique SHA256: 10000
Duplicate SHA256: 0

Label distribution:
label
1    5000
0    5000
Name: count, dtype: int64

Columns:
['md5', 'sha1', 'sha256', 'tlsh', 'first_submission_date', 'last_analysis_date', 'detection_ratio', 'label', 'file_type', 'family', 'family_confidence', 'behavior', 'file_property', 'packer', 'exploit', 'group', 'histogram', 'byteentropy', 'strings', 'general', 'header', 'section', 'imports', 'exports', 'datadirectories', 'richheader', 'authenticode', 'pefilewarnings', 'week_id', 'caps', 'ttps', 'mbc']


Step 3. Verify the dataset

In [19]:

assert len(df) == 10000
assert df["sha256"].nunique() == 10000
assert df["sha256"].duplicated().sum() == 0

print("Dataset integrity verified.")
print("Ready for structured parsing.")


Dataset integrity verified.
Ready for structured parsing.


Next: Step 2 – Identify structured and serialized fields
Before parsing anything, we need to understand the actual Python data types and formats.


## Step 2 – Identify Nested and Serialized Fields

### Objective
Inspect all dataset columns to identify:
- Native Python lists and dictionaries
- JSON strings
- Python literal strings
- Plain text
- Empty lists and missing values
- Malformed serialized values

We will determine the correct parsing strategy for each field before transforming the dataset.


In [20]:

# ==========================================
# STEP 2: STRUCTURED FIELD IDENTIFICATION
# ==========================================

from collections import Counter

def identify_type(value):

    if value is None:
        return "None"

    if isinstance(value, float) and np.isnan(value):
        return "NaN"

    if isinstance(value, dict):
        return "Native Dictionary"

    if isinstance(value, list):
        return "Native List"

    if isinstance(value, tuple):
        return "Native Tuple"

    if isinstance(value, str):
        stripped = value.strip()

        if not stripped:
            return "Empty String"

        try:
            json.loads(stripped)
            return "JSON String"
        except (json.JSONDecodeError, TypeError):
            pass

        try:
            ast.literal_eval(stripped)
            return "Python Literal String"
        except (ValueError, SyntaxError, TypeError):
            return "Plain Text / Other String"

    return type(value).__name__


# Inspect every column
type_report = []

for col in df.columns:

    counts = Counter(
        identify_type(value)
        for value in df[col]
    )

    for data_type, count in counts.items():

        type_report.append({
            "column": col,
            "data_type": data_type,
            "count": count,
            "percentage": round(
                count / len(df) * 100, 2
            )
        })

type_report_df = pd.DataFrame(type_report)

print("STRUCTURED DATA TYPE REPORT")
display(type_report_df)


STRUCTURED DATA TYPE REPORT


,column,data_type,count,percentage
0,md5,Plain Text / Other String,10000,100.00
1,sha1,Plain Text / Other String,10000,100.00
2,sha256,Plain Text / Other String,10000,100.00
3,tlsh,Plain Text / Other String,10000,100.00
4,first_submission_date,int,10000,100.00
5,last_analysis_date,int,10000,100.00
6,detection_ratio,Plain Text / Other String,10000,100.00
7,label,int,10000,100.00
8,file_type,Plain Text / Other String,10000,100.00
9,family,NaN,6036,60.36


. Analysis of your output
Your dataset contains three important types of data:

A. Already structured data
Columns	Current type
caps	Native List
ttps	Native List
mbc	Native List

These three columns already contain Python lists. We should not unnecessarily parse them again.

B. JSON serialized fields

Examples:
- behavior – 9,211 JSON strings
- file_property – 9,873 JSON strings
- packer – 9,621 JSON strings
- exploit – 9,995 JSON strings
- group – 8,037 JSON strings
- histogram – 10,000 JSON strings
- byteentropy – 10,000 JSON strings
C. Python literal serialized fields

Examples:
- strings – 10,000 Python literal strings
- general – 10,000 Python literal strings
- header – 9,998 Python literal strings
- section – 9,998 Python literal strings
- imports – 7,332 Python literal strings
- exports – 4,102 Python literal strings
- datadirectories – 9,998 Python literal strings
There are also mixed-format fields, such as behavior, file_property, and packer.

Important finding: We need two parsing methods: json.loads() and ast.literal_eval(). We must not use eval().

2. Next step – Inspect actual nested structures
Before building our parsing engine, we need to inspect representative records.


## Step 3 – Inspect Actual Nested Structures

### Objective
Inspect representative values from every serialized and nested field.

We will:
1. Examine actual JSON structures.
2. Examine Python literal structures.
3. Identify dictionaries, lists and nested objects.
4. Identify empty structures.
5. Identify malformed serialized values.
6. Determine field-specific parsing requirements.

No dataset transformations will be performed in this step.


What I want to verify
Pay particular attention to:
- Whether imports and exports contain nested dictionaries.
- Whether behavior contains MITRE-like behavior descriptions.
- Whether caps, ttps, and mbc contain meaningful threat intelligence.
- Whether general, header, and section contain PE file metadata.
- Whether strings contains extracted strings useful for malware analysis.

In [31]:

# ==========================================
# STEP 3: ACTUAL NESTED STRUCTURE INSPECTION
# ==========================================

import json
import ast
import pandas as pd

nested_columns = [
    "behavior",
    "file_property",
    "packer",
    "exploit",
    "group",
    "histogram",
    "byteentropy",
    "strings",
    "general",
    "header",
    "section",
    "imports",
    "exports",
    "datadirectories",
    "richheader",
    "authenticode",
    "pefilewarnings",
    "caps",
    "ttps",
    "mbc"
]

def safe_inspect(value):

    if not isinstance(value, str):
        return value

    try:
        return json.loads(value)
    except (ValueError, TypeError):
        pass

    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError, TypeError):
        return value


for col in nested_columns:

    print("\n" + "=" * 80)
    print("COLUMN:", col.upper())
    print("=" * 80)

    if col not in df.columns:
        print("Column not found")
        continue

    samples = df[col].dropna().head(2)

    for index, value in samples.items():

        parsed = safe_inspect(value)

        print("\nRecord index:", index)
        print("Original type:", type(value).__name__)
        print("Parsed type:", type(parsed).__name__)

        print("Sample:")

        print(
            str(parsed)[:700]
        )



COLUMN: BEHAVIOR

Record index: 0
Original type: str
Parsed type: list
Sample:
[]

Record index: 1
Original type: str
Parsed type: list
Sample:
[]

COLUMN: FILE_PROPERTY

Record index: 0
Original type: str
Parsed type: list
Sample:
[]

Record index: 1
Original type: str
Parsed type: list
Sample:
[]

COLUMN: PACKER

Record index: 0
Original type: str
Parsed type: list
Sample:
[]

Record index: 1
Original type: str
Parsed type: list
Sample:
[]

COLUMN: EXPLOIT

Record index: 0
Original type: str
Parsed type: list
Sample:
[]

Record index: 1
Original type: str
Parsed type: list
Sample:
[]

COLUMN: GROUP

Record index: 0
Original type: str
Parsed type: list
Sample:
[]

Record index: 1
Original type: str
Parsed type: list
Sample:
[]

COLUMN: HISTOGRAM

Record index: 0
Original type: str
Parsed type: list
Sample:
[121643, 50401, 47908, 51367, 44484, 46537, 46618, 48306, 45207, 45353, 45649, 47311, 45663, 47397, 47391, 52064, 45377, 45346, 44593, 45818, 46019, 45967, 46180, 47768, 46480, 462

Step 3 – Additional investigation
Let's run a better inspection that finds the first non-empty value in each column.

In [32]:

# ==========================================
# STEP 3B: FIND NON-EMPTY NESTED STRUCTURES
# ==========================================

def is_empty_structure(value):

    if value is None:
        return True

    if isinstance(value, float) and np.isnan(value):
        return True

    if isinstance(value, (list, dict, tuple, str)):
        return len(value) == 0 or (
            isinstance(value, str) and not value.strip()
        )

    return False


inspection_results = []

for col in nested_columns:

    if col not in df.columns:
        continue

    found = False

    for idx, value in df[col].items():

        parsed = safe_inspect(value)

        if is_empty_structure(parsed):
            continue

        inspection_results.append({
            "column": col,
            "record_index": idx,
            "original_type": type(value).__name__,
            "parsed_type": type(parsed).__name__,
            "sample": str(parsed)[:350]
        })

        found = True
        break

    if not found:
        inspection_results.append({
            "column": col,
            "record_index": None,
            "original_type": None,
            "parsed_type": "No non-empty sample",
            "sample": None
        })

inspection_df = pd.DataFrame(inspection_results)

display(inspection_df)


,column,record_index,original_type,parsed_type,sample
0,behavior,6,str,list,['backdoor']
1,file_property,124,str,list,['msil']
2,packer,14,str,list,['vmprotect']
3,exploit,124,str,list,"['cve_2019_18935', 'cve_2019_189']"
4,group,24,str,list,['gamaredon']
5,histogram,0,str,list,"[121643, 50401, 47908, 51367, 44484, 46537, 46..."
6,byteentropy,0,str,list,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
7,strings,0,str,dict,"{'numstrings': 56656, 'avlength': 6.6226701496..."
8,general,0,str,dict,"{'size': 12252151, 'entropy': 7.99358046077606..."
9,header,0,str,dict,"{'coff': {'timestamp': 1726319835, 'machine': ..."


Conclusion -- What your output tells us
Your dataset contains three different kinds of nested structures.
Columns	Actual structure	Treatment
behavior	List	Parse and inspect elements
file_property	List	Parse and inspect elements
packer	List	Parse and inspect elements
exploit	List	Parse and inspect elements
group	List	Parse and inspect elements
histogram	List	Numeric feature extraction
byteentropy	List	Numeric feature extraction
strings	Dictionary	Extract metadata
general	Dictionary	Extract PE metadata
header	Dictionary	Extract PE header information
section	Dictionary	Extract section information
imports	Dictionary	Extract DLL/API information
exports	List	Extract exported functions
datadirectories	List	Extract PE directory information
richheader	List	Inspect compiler-related metadata
authenticode	Dictionary	Extract signature information
pefilewarnings	List	Preserve and investigate warnings
caps, ttps, mbc	Native lists	Inspect directly

A particularly important observation:
- caps, ttps, and mbc are already Python lists. They don't need string deserialization.
- strings, general, header, section, and imports are dictionaries after parsing.
- pefilewarnings contains parsing-warning messages. We must not mistake these messages for valid malware features.
Our next objective
Before modifying anything, we need to understand the internal keys and elements of these structures.

For example:
- general → file size, entropy, architecture, etc.
- imports → DLL names and imported API functions.
- behavior → malware behavioral indicators.
- ttps → MITRE ATT&CK-related information.
We should not flatten everything into columns blindly. Some information should remain nested, while selected information can become ML features.

Step 4 – Profile internal structures
Create a new Markdown cell:

Step 4 – Inspect Internal Dictionary Keys and List Elements
This examines up to three non-empty records per nested column rather than assuming the first record represents the entire dataset.

In [33]:

# ==========================================
# STEP 4: INTERNAL STRUCTURE PROFILING
# ==========================================

structure_profile = []

for col in nested_columns:

    if col not in df.columns:
        continue

    inspected = 0

    for idx, value in df[col].items():

        parsed = safe_inspect(value)

        if is_empty_structure(parsed):
            continue

        if isinstance(parsed, dict):

            structure_profile.append({
                "column": col,
                "record_index": idx,
                "structure": "Dictionary",
                "keys": list(parsed.keys())[:20],
                "list_length": None,
                "first_item_type": None,
                "first_item_keys": None
            })

        elif isinstance(parsed, list):

            first_item = parsed[0] if parsed else None

            structure_profile.append({
                "column": col,
                "record_index": idx,
                "structure": "List",
                "keys": None,
                "list_length": len(parsed),
                "first_item_type": (
                    type(first_item).__name__
                    if first_item is not None else None
                ),
                "first_item_keys": (
                    list(first_item.keys())[:20]
                    if isinstance(first_item, dict) else None
                )
            })

        inspected += 1

        if inspected >= 3:
            break

structure_profile_df = pd.DataFrame(structure_profile)

display(structure_profile_df)


,column,record_index,structure,keys,list_length,first_item_type,first_item_keys
0,behavior,6,List,None,1.0,str,None
1,behavior,11,List,None,1.0,str,None
2,behavior,16,List,None,1.0,str,None
3,file_property,124,List,None,1.0,str,None
4,file_property,148,List,None,1.0,str,None
5,file_property,152,List,None,1.0,str,None
6,packer,14,List,None,1.0,str,None
7,packer,73,List,None,1.0,str,None
8,packer,120,List,None,1.0,str,None
9,exploit,124,List,None,2.0,str,None


Conclusion - 1. Findings from your output
Step 4 completed
Field	Actual structure	What we need to extract
behavior	List of strings	Behavioral indicators
file_property	List of strings	File characteristics
packer	List of strings	Packing information
exploit	List of strings	CVE identifiers
group	List of strings	Malware group names
histogram	256 integers	Byte frequency distribution
byteentropy	256 integers	Entropy distribution
strings	Dictionary	Extracted string statistics
general	Dictionary	PE metadata
header	Dictionary	PE header information
section	Dictionary	Section metadata
imports	Dictionary	DLLs and imported APIs
exports	List	Exported functions
datadirectories	List of dictionaries	PE directory metadata
richheader	List	Compiler metadata
authenticode	Dictionary	Digital signature metadata
pefilewarnings	List	Parsing warnings
caps	List of dictionaries	Malware capabilities
ttps	List of dictionaries	MITRE ATT&CK tactics and techniques
mbc	List of dictionaries	Malware Behavior Catalog information

Important discovery: caps, ttps, and mbc contain dictionaries inside lists. These are particularly valuable for our SOC platform because they provide security-related context.

2. Step 5 – Inspect actual values and nested keys
Now we need to inspect the contents, not just the data types.

Create a Markdown cell:

Step 5 – Deep Inspection of Representative Nested Values

Then add this code cell.
3. What we are trying to achieve
For example, we want to understand whether an import record looks like this:
{
    "KERNEL32.dll": [
        "CreateProcessW",
        "WriteFile",
        "ReadFile"
    ],
    "USER32.dll": [
        "MessageBoxW"
    ]
}
And whether a MITRE-related record looks like this:
{
    "Tactic": "EXECUTION",
    "Technique": "Shared Modules"
}
These are illustrative examples, not assumed actual records.

The actual output will determine our extraction logic.

In [34]:

# ==========================================
# STEP 5: DEEP INSPECTION OF NESTED VALUES
# ==========================================

from pprint import pprint

def preview_structure(value, max_items=3):

    if isinstance(value, dict):
        return {
            key: (
                str(val)[:150]
                if not isinstance(val, (dict, list))
                else f"{type(val).__name__} ({len(val)} items)"
            )
            for key, val in list(value.items())[:max_items]
        }

    elif isinstance(value, list):
        return {
            "length": len(value),
            "first_elements": value[:max_items]
        }

    return str(value)[:300]


# Fields to investigate
inspect_columns = [
    "behavior", "file_property", "packer",
    "exploit", "group", "histogram", "byteentropy",
    "strings", "general", "header", "section",
    "imports", "exports", "datadirectories",
    "richheader", "authenticode", "pefilewarnings",
    "caps", "ttps", "mbc"
]

for col in inspect_columns:

    if col not in df.columns:
        continue

    sample = None
    sample_index = None

    for idx, value in df[col].items():

        parsed = safe_inspect(value)

        if not is_empty_structure(parsed):
            sample = parsed
            sample_index = idx
            break

    print("\n" + "=" * 70)
    print(f"COLUMN: {col.upper()}")
    print("=" * 70)

    if sample is None:
        print("No non-empty sample found.")
        continue

    print("Record index:", sample_index)
    print("Structure:", type(sample).__name__)

    pprint(preview_structure(sample))



COLUMN: BEHAVIOR
Record index: 6
Structure: list
{'first_elements': ['backdoor'], 'length': 1}

COLUMN: FILE_PROPERTY
Record index: 124
Structure: list
{'first_elements': ['msil'], 'length': 1}

COLUMN: PACKER
Record index: 14
Structure: list
{'first_elements': ['vmprotect'], 'length': 1}

COLUMN: EXPLOIT
Record index: 124
Structure: list
{'first_elements': ['cve_2019_18935', 'cve_2019_189'], 'length': 2}

COLUMN: GROUP
Record index: 24
Structure: list
{'first_elements': ['gamaredon'], 'length': 1}

COLUMN: HISTOGRAM
Record index: 0
Structure: list
{'first_elements': [121643, 50401, 47908], 'length': 256}

COLUMN: BYTEENTROPY
Record index: 0
Structure: list
{'first_elements': [0, 0, 0], 'length': 256}

COLUMN: STRINGS
Record index: 0
Structure: dict
{'avlength': '6.622670149675233',
 'numstrings': '56656',
 'printabledist': 'list (96 items)'}

COLUMN: GENERAL
Record index: 0
Structure: dict
{'entropy': '7.993580460776063', 'is_pe': '1', 'size': '12252151'}

COLUMN: HEADER
Record index

Good. Step 5 is running, and we have already confirmed several actual malware indicators.

From your screenshot:
- behavior: backdoor
- file_property: msil
- packer: vmprotect
- mbc: contains structured dictionaries with Objective and Behavior.
- exploit: contains exploit-related identifiers.
These are useful features for malware classification and SOC investigation.

However, the output is truncated, so I don't want us to design parsers without seeing the important fields.

Step 5.1 – Focused inspection
Create another code cell below Step 5. This time, inspect the most important fields individually.
Why this step matters
We need to identify exactly which fields can become useful detection features.

For example:
Field	Potential SOC application
imports	Identify suspicious API imports
general	PE metadata and entropy analysis
strings	Detect suspicious embedded strings
caps	Extract malware capabilities
ttps	MITRE ATT&CK mapping
mbc	Malware behavior classification
authenticode	Digital signature analysis

One important distinction: An imported API or a high-entropy value alone does not prove malware. We'll use multiple indicators together.

In [35]:

# ==========================================
# STEP 5.1: FOCUSED NESTED FIELD INSPECTION
# ==========================================

important_columns = [
    "strings",
    "general",
    "header",
    "section",
    "imports",
    "caps",
    "ttps",
    "mbc",
    "authenticode"
]

for col in important_columns:

    print("\n" + "=" * 65)
    print(f"FIELD: {col.upper()}")
    print("=" * 65)

    count = 0

    for idx, value in df[col].items():

        parsed = safe_inspect(value)

        if is_empty_structure(parsed):
            continue

        print(f"\nRecord index: {idx}")
        print("Python type:", type(parsed).__name__)

        if isinstance(parsed, dict):

            print("Dictionary keys:", list(parsed.keys()))

            for key, val in list(parsed.items())[:5]:
                print(f"  {key}: {str(val)[:250]}")

        elif isinstance(parsed, list):

            print("List length:", len(parsed))
            print("First two elements:")

            pprint(parsed[:2])

        count += 1

        if count >= 2:
            break



FIELD: STRINGS

Record index: 0
Python type: dict
Dictionary keys: ['numstrings', 'avlength', 'printabledist', 'printables', 'entropy', 'string_counts']
  numstrings: 56656
  avlength: 6.622670149675233
  printabledist: [4017, 3063, 3271, 3212, 3518, 3260, 3140, 3284, 3252, 3882, 3096, 3190, 3249, 3857, 4994, 3312, 3363, 3305, 3315, 3211, 3373, 3197, 3231, 3186, 3164, 3383, 3279, 3323, 3363, 3258, 3311, 3364, 2932, 4413, 3438, 3369, 3389, 3829, 3239, 3403, 3512, 361
  printables: 375214
  entropy: 6.52505596905308

Record index: 1
Python type: dict
Dictionary keys: ['numstrings', 'avlength', 'printabledist', 'printables', 'entropy', 'string_counts']
  numstrings: 3603
  avlength: 14.485983902303635
  printabledist: [2247, 27, 62, 21, 1390, 83, 20, 48, 210, 95, 24, 328, 62, 182, 427, 125, 733, 290, 198, 436, 178, 136, 166, 93, 358, 341, 221, 406, 101, 60, 98, 527, 2016, 2826, 147, 379, 721, 491, 230, 85, 1599, 232, 51, 41, 422, 109, 63, 55, 306, 112, 168, 373, 4
  printables: 52193
  e

1. Findings from your output
A. strings – Dictionary
Your dataset contains:
- numstrings: 56,656
- avlength: 6.62
- printabledist: Distribution of printable characters
- printables: 375,214
- entropy: 6.525
These are useful for static malware analysis.

For example, entropy can help identify potentially packed or encrypted content, although entropy alone cannot establish that a file is malicious.
B. general – Dictionary
Your output shows fields such as:
- self_signed
- empty_program_name
- no_countersigner
- parse_error
These can help us investigate executable metadata and signature-related characteristics.

2. One more inspection before actual parsing
I want to inspect the complete structure of these fields without producing enormous outputs.

In [36]:

# ==========================================
# STEP 5.2: COMPACT STRUCTURE INSPECTION
# ==========================================

inspect_columns = [
    "strings",
    "general",
    "header",
    "section",
    "imports",
    "caps",
    "ttps",
    "mbc",
    "authenticode"
]

for col in inspect_columns:

    print("\n" + "=" * 60)
    print("COLUMN:", col.upper())
    print("=" * 60)

    for idx, value in df[col].items():

        parsed = safe_inspect(value)

        if is_empty_structure(parsed):
            continue

        print("Record:", idx)
        print("Type:", type(parsed).__name__)

        if isinstance(parsed, dict):
            print("Keys:", list(parsed.keys()))

            for key, val in list(parsed.items())[:10]:
                print(f"{key}: {str(val)[:150]}")

        elif isinstance(parsed, list):
            print("Length:", len(parsed))
            print("First item:", str(parsed[0])[:300]
                  if parsed else "Empty")

        break



COLUMN: STRINGS
Record: 0
Type: dict
Keys: ['numstrings', 'avlength', 'printabledist', 'printables', 'entropy', 'string_counts']
numstrings: 56656
avlength: 6.622670149675233
printabledist: [4017, 3063, 3271, 3212, 3518, 3260, 3140, 3284, 3252, 3882, 3096, 3190, 3249, 3857, 4994, 3312, 3363, 3305, 3315, 3211, 3373, 3197, 3231, 3186, 3164,
printables: 375214
entropy: 6.52505596905308
string_counts: {'base64': 2, 'command': 50, 'connect': 8, 'cookie': 3, 'create': 14, 'crypt': 1, 'debug': 4, 'decode': 4, 'delete': 8, 'directory': 15, 'dos_msg': 1,

COLUMN: GENERAL
Record: 0
Type: dict
Keys: ['size', 'entropy', 'is_pe', 'start_bytes']
size: 12252151
entropy: 7.993580460776063
is_pe: 1
start_bytes: [77, 90, 144, 0]

COLUMN: HEADER
Record: 0
Type: dict
Keys: ['coff', 'optional', 'dos']
coff: {'timestamp': 1726319835, 'machine': 'IMAGE_FILE_MACHINE_AMD64', 'number_of_sections': 6, 'number_of_symbols': 0, 'sizeof_optional_header': 240, 'poin
optional: {'magic': 523, 'subsystem': 'IMAGE_SUBSY

Excellent. This is exactly what we wanted to discover. Your inspection confirms that the dataset contains genuine nested dictionaries and lists, not just JSON strings.

We can now move from inspection to building reusable parsing functions.
1. What we have confirmed
Field	Actual structure	Important information
strings	Dictionary	String counts, entropy, printable characters
general	Dictionary	File size, PE indicator, entropy, start bytes
header	Dictionary	Executable header information
section	Dictionary	PE section information
imports	Dictionary	Imported DLLs and functions
exports	List	Exported functions
caps	List of dictionaries	Malware capabilities
ttps	List of dictionaries	MITRE tactics and techniques
mbc	List of dictionaries	Malware behavior classification

These fields will be valuable for our malware feature extraction and detection pipeline.

2. Next: Build reusable parsing functions
Step 6 – Implementation
Add a Markdown cell:

Step 6 – Build Reusable Structured Data Parsers

Then add this code cell.

In [37]:

# ==========================================
# STEP 6: REUSABLE STRUCTURED DATA PARSERS
# ==========================================

import json
import ast
import pandas as pd
import numpy as np

# Nested columns identified during inspection

nested_columns = [
    "behavior",
    "file_property",
    "packer",
    "exploit",
    "group",
    "histogram",
    "byteentropy",
    "strings",
    "general",
    "header",
    "section",
    "imports",
    "exports",
    "datadirectories",
    "richheader",
    "authenticode",
    "pefilewarnings",
    "caps",
    "ttps",
    "mbc"
]

# Preserve original dataset
df_structured = df.copy()

# Check available columns
nested_columns = [
    col for col in nested_columns
    if col in df_structured.columns
]


# ==========================================
# 1. MISSING VALUE DETECTION
# ==========================================

def is_missing_value(value):

    if value is None:
        return True

    if isinstance(value, (float, np.floating)):
        return bool(np.isnan(value))

    return False


# ==========================================
# 2. UNIVERSAL STRUCTURED PARSER
# ==========================================

def parse_structured_value(value):

    if is_missing_value(value):
        return value, "missing"

    # Already structured
    if isinstance(value, (dict, list)):
        return value, "already_structured"

    # Parse string values
    if isinstance(value, str):

        if not value.strip():
            return value, "empty_string"

        # Attempt JSON parsing
        try:
            parsed = json.loads(value)
            return parsed, "json_parsed"

        except (json.JSONDecodeError, TypeError):
            pass

        # Attempt Python literal parsing
        try:
            parsed = ast.literal_eval(value)
            return parsed, "python_literal_parsed"

        except (ValueError, SyntaxError, TypeError):
            pass

        # Preserve unparseable original value
        return value, "unparsed_string"

    return value, "other_type"


# ==========================================
# 3. APPLY PARSER WITHOUT LOSING RAW DATA
# ==========================================

parse_audit = {}

for col in nested_columns:

    parsed_results = df_structured[col].apply(
        parse_structured_value
    )

    df_structured[col] = parsed_results.apply(
        lambda result: result[0]
    )

    parse_audit[col] = parsed_results.apply(
        lambda result: result[1]
    )


print("STRUCTURED PARSING COMPLETED")
print("Records:", len(df_structured))
print("Nested columns processed:", len(nested_columns))
print("Original dataset remains unchanged.")


STRUCTURED PARSING COMPLETED
Records: 10000
Nested columns processed: 20
Original dataset remains unchanged.


3. Generate parsing audit report

In [38]:

# ==========================================
# STEP 7: PARSING AUDIT REPORT
# ==========================================

audit_report = pd.DataFrame(parse_audit)

audit_summary = []

for col in nested_columns:

    counts = audit_report[col].value_counts()

    for status, count in counts.items():

        audit_summary.append({
            "column": col,
            "status": status,
            "count": count,
            "percentage": round(
                (count / len(df_structured)) * 100, 2
            )
        })

audit_summary = pd.DataFrame(audit_summary)

display(audit_summary)

print("\nTOTAL PARSING STATUS COUNTS:")
display(audit_report.apply(pd.Series.value_counts).fillna(0).astype(int))


,column,status,count,percentage
0,behavior,json_parsed,9211,92.11
1,behavior,python_literal_parsed,789,7.89
2,file_property,json_parsed,9873,98.73
3,file_property,python_literal_parsed,127,1.27
4,packer,json_parsed,9621,96.21
5,packer,python_literal_parsed,379,3.79
6,exploit,json_parsed,9995,99.95
7,exploit,python_literal_parsed,5,0.05
8,group,json_parsed,8037,80.37
9,group,python_literal_parsed,1963,19.63



TOTAL PARSING STATUS COUNTS:


,behavior,file_property,packer,exploit,group,histogram,byteentropy,strings,general,header,section,imports,exports,datadirectories,richheader,authenticode,pefilewarnings,caps,ttps,mbc
already_structured,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,10000,10000,10000
json_parsed,9211,9873,9621,9995,8037,10000,10000,0,0,2,2,2668,5898,2,10000,2,3498,0,0,0
python_literal_parsed,789,127,379,5,1963,0,0,10000,10000,9998,9998,7332,4102,9998,0,9998,6502,0,0,0


Step 8 – Final Structured Parsing Validation

In [39]:

# ==========================================
# STEP 8: FINAL PARSING VALIDATION
# ==========================================

# 1. Verify record count
assert len(df_structured) == 10000

# 2. Verify original dataset remains unchanged
assert len(df) == 10000

# 3. Verify class distribution
print("ORIGINAL LABEL DISTRIBUTION:")
print(df["label"].value_counts())

print("\nSTRUCTURED LABEL DISTRIBUTION:")
print(df_structured["label"].value_counts())

assert df["label"].equals(df_structured["label"])

# 4. Verify SHA256 integrity
assert df["sha256"].equals(df_structured["sha256"])

assert df_structured["sha256"].nunique() == 10000

# 5. Verify no unparsed strings remain
unparsed_counts = {}

for col in nested_columns:
    unparsed_counts[col] = (
        audit_report[col] == "unparsed_string"
    ).sum()

print("\nUNPARSED VALUES:")
print(pd.Series(unparsed_counts))

assert sum(unparsed_counts.values()) == 0

# 6. Verify dataset dimensions
print("\nFINAL DATASET SHAPE:", df_structured.shape)

print("\nALL VALIDATION CHECKS PASSED.")


ORIGINAL LABEL DISTRIBUTION:
label
1    5000
0    5000
Name: count, dtype: int64

STRUCTURED LABEL DISTRIBUTION:
label
1    5000
0    5000
Name: count, dtype: int64

UNPARSED VALUES:
behavior           0
file_property      0
packer             0
exploit            0
group              0
histogram          0
byteentropy        0
strings            0
general            0
header             0
section            0
imports            0
exports            0
datadirectories    0
richheader         0
authenticode       0
pefilewarnings     0
caps               0
ttps               0
mbc                0
dtype: int64

FINAL DATASET SHAPE: (10000, 32)

ALL VALIDATION CHECKS PASSED.


1. Validation results
Validation	Result
Original records	10,000
Structured records	10,000
Original label distribution	5,000 / 5,000
Structured label distribution	5,000 / 5,000
SHA256 integrity	Passed
Unparsed nested values	0
Final dataset dimensions	10,000 × 32

Your dataset remains balanced:
- Label 1: 5,000 records.
- Label 0: 5,000 records.

2. Next phase: Nested Feature Extraction
Now we move from parsing data to extracting meaningful malware characteristics.

Our objective is to convert complex nested structures into useful features that machine-learning algorithms can understand.

For example:
Original nested data
{
    "size": 12252151,
    "entropy": 7.9935,
    "is_pe": 1,
    "start_bytes": [77, 90, 144, 0]
}

Extracted numerical features
Feature	Value
general_size	12252151
general_entropy	7.9935
general_is_pe	1
start_byte_0	77
start_byte_1	90
start_byte_2	144
start_byte_3	0

. Feature extraction roadmap
We will process the following groups:
Group	Features to extract
General	File size, entropy, PE status
Strings	Number of strings, average length, printable characters, suspicious keywords
Header	COFF characteristics, optional header, DOS information
Section	Section count, names, sizes, entropy
Imports	DLL count, API count, suspicious API indicators
Exports	Export count and names
Histogram	Byte frequency distribution
Byteentropy	Byte entropy distribution
Behavior	Behavioral indicators
File property	File properties and metadata
Packer	Packer indicators
Exploit / Group	Exploit and malware group indicators
Capabilities	Capability indicators
TTPS / MBC	MITRE ATT&CK-related tactics, techniques and behaviors

We will also preserve useful original metadata such as family, file_type, and detection_ratio for analysis.

Important: We will not blindly flatten everything. We will retain meaningful information, avoid unnecessary feature explosion, and prevent label leakage.

Step 9 – Nested Feature Extraction

Our first task will be to inspect the parsed structures and establish a feature extraction strategy before transforming all 10,000 records.

We will maintain three separate datasets:
1. df – Original dataset.
2. df_structured – Parsed dataset.
3. df_features – Extracted, ML-ready candidate features.

In [40]:
from pathlib import Path

# Check whether the structured dataset exists
print("Available dataset variables:")

for name in ["df_structured", "df_parsed", "df"]:
    if name in globals():
        obj = globals()[name]
        print(name, type(obj), getattr(obj, "shape", None))

Available dataset variables:
df_structured <class 'pandas.DataFrame'> (10000, 32)
df <class 'pandas.DataFrame'> (10000, 32)


Step 1 – Export the structured dataset
Go back to 02_dataset_1_structured_parsing.ipynb.

Add a Markdown cell:

## Step 9 – Export Structured Dataset

Export the validated structured dataset for reuse
in the feature extraction phase.

In [41]:
from pathlib import Path
import pickle

# Create output directory
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Export structured dataset
output_path = output_dir / "dataset1_structured.pkl"

with open(output_path, "wb") as file:
    pickle.dump(df_structured, file)

print("STRUCTURED DATASET EXPORTED SUCCESSFULLY")
print("File:", output_path)
print("Shape:", df_structured.shape)
print("File size (MB):", round(output_path.stat().st_size / (1024 * 1024), 2))

STRUCTURED DATASET EXPORTED SUCCESSFULLY
File: data/processed/dataset1_structured.pkl
Shape: (10000, 32)
File size (MB): 201.17
